In [83]:
import pandas as pd
import numpy as np
import os

# Training datasets

### Players DF reduced without context games

In [ ]:
year = 2024

players_reduced_df = pd.read_csv(f"procesed_data/season_player_stats_{year}_reduced/all_players.csv", encoding='utf-8')
players_historical_df = pd.read_csv(f"procesed_data/player_mean_stats_reduced.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)
"""

pca_cols_current = [c for c in players_reduced_df.columns if c.startswith('PCA')]
pca_cols_historic = [c for c in players_historical_df.columns if c.startswith('PCA')]

for game_id, game_df in players_reduced_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for team, team_df in game_df.groupby('team'):
        # Team home
        if team_df.iloc[0]['location'] == 'Home':
            prefix = "th"
            if team_df.iloc[0]["win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "ta"
        
        team_df = team_df.sort_values(by='mp', ascending=False)
        top5_mp = team_df.head(5)['mp'].sum()
        bench_mp = team_df['mp'].sum() - top5_mp
        row[f"{prefix}_bench_usg"] = bench_mp / top5_mp if top5_mp > 0 else 0
        # Get the 7 players with the most minutes played
        for player_idx in range(min(7, len(team_df))):  # Asegura máximo 7 jugadores
            player = team_df.iloc[player_idx]
            player_count = player_idx + 1
            for col in pca_cols_current:
                row[f"{prefix}_player{player_count}_{col}"] = player[col]
            
            # Stats históricos (con manejo de errores)
            player_name = player['player']
            hist_data = players_historical_df[players_historical_df['Player'] == player_name]
            
            for col in pca_cols_historic:
                if not hist_data.empty:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = hist_data[col].values[0]
                else:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = 0
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)

if not os.path.exists(f"final_data/season_players_dataset_{year}_reduced"):
    os.makedirs(f"final_data/season_players_dataset_{year}_reduced")
# Guardar resultado
final_dataset.to_csv(f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Players DF without context games

In [ ]:
year = 2024

players_df = pd.read_csv(f"raw_data/season_player_stats_{year}/all_players.csv", encoding='utf-8')
players_historical_df = pd.read_csv(f"procesed_data/player_mean_stats.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)
"""

stats_cols_current = [c for c in players_df.columns if not c in ["player","team","game_id","location","opponent","team_score","opponent_score","win","played","mp"]]
stats_cols_historic = [c for c in players_historical_df.columns if not c in ["Player"]]

for game_id, game_df in players_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for team, team_df in game_df.groupby('team'):
        # Team home
        if team_df.iloc[0]['location'] == 'Home':
            prefix = "th"
            if team_df.iloc[0]["win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "ta"
        
        team_df = team_df.sort_values(by='mp', ascending=False)
        top5_mp = team_df.head(5)['mp'].sum()
        bench_mp = team_df['mp'].sum() - top5_mp
        row[f"{prefix}_bench_usg"] = bench_mp / top5_mp if top5_mp > 0 else 0
        # Get the 7 players with the most minutes played
        for player_idx in range(min(7, len(team_df))):  # Asegura máximo 7 jugadores
            player = team_df.iloc[player_idx]
            player_count = player_idx + 1
            for col in stats_cols_current:
                row[f"{prefix}_player{player_count}_{col}"] = player[col]
            
            # Stats históricos (con manejo de errores)
            player_name = player['player']
            hist_data = players_historical_df[players_historical_df['Player'] == player_name]
            
            for col in stats_cols_historic:
                if not hist_data.empty:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = hist_data[col].values[0]
                else:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = 0
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)

if not os.path.exists(f"final_data/season_players_dataset_{year}"):
    os.makedirs(f"final_data/season_players_dataset_{year}")
# Guardar resultado
final_dataset.to_csv(f"final_data/season_players_dataset_{year}/season_players_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF without context games

In [ ]:
year = 2024

teams_df = pd.read_csv(f"raw_data/season_team_stats_{year}/all_teams.csv", encoding='utf-8')
teams_historical_df = pd.read_csv(f"raw_data/all_teams_data.csv", encoding='utf-8')
teams_historical_df = teams_historical_df[teams_historical_df["Year"]>= 2020]
final_dataset = pd.DataFrame()

"""
team_home_stat1, ..., team_home_statN, team_home_historic_stat1_1, ..., team_home_historic_statM_Z, 
team_away_stat1, ..., team_away_statN, team_away_historic_stat1_1, ..., team_away_historic_statM_Z, team_home_wins (1 o 0)
"""

cols_current = [c for c in teams_df.columns if not c.startswith('opp')]
cols_historic = [c for c in teams_historical_df.columns if not c in ["Year", "Team"]]

# teams_df = teams_df.rename(columns={col: f"team_home_{col}" for col in cols_current})
# teams_df = teams_df.rename(columns={col: f"team_away_{col}" for col in cols_current})

for game_id, game_df in teams_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for location, team_df in game_df.groupby('Location'):
        # Team home
        if location == 'Home':
            prefix = "team_home"
            if team_df.iloc[0]["Win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "team_away"
        
        for col in cols_current:
            if col == 'game_id' or col == 'Location' or col == 'Win' or col == "Opponent" or col == "team":
                continue
            row[f"{prefix}_{col}"] = team_df[col].values[0]
        
        team_historical_df = teams_historical_df[teams_historical_df['Team'] == team_df.iloc[0]['team']]
        for team_year in team_historical_df['Year'].unique():
            for col in cols_historic:
                row[f"{prefix}_{str(team_year)}_historic_{col}"] = team_historical_df[col].values[0]
        
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)

if not os.path.exists(f"final_data/season_teams_dataset_{year}"):
    os.makedirs(f"final_data/season_teams_dataset_{year}")
# Guardar resultado
final_dataset.to_csv(f"final_data/season_teams_dataset_{year}/season_teams_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF reduced without context games

In [ ]:
year = 2024

teams_df = pd.read_csv(f"procesed_data/season_team_stats_{year}_reduced/all_teams.csv", encoding='utf-8')
teams_historical_df = pd.read_csv(f"procesed_data/all_teams_data_reduced.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
team_home_component1, ..., team_home_componentN_K, team_home_historic_component1_1, ..., team_home_historic_componentM_Z, 
team_away_component1, ..., team_away_componentN_K, team_away_historic_component1_1, ..., team_away_historic_componentM_Z, team_home_wins (1 o 0)
"""

pca_cols_current = [c for c in teams_df.columns if not (c.startswith('PCA6') or c.startswith('PCA7') or c.startswith('PCA8'))] # Components of the opponent are excluded
pca_cols_historic = [c for c in teams_historical_df.columns if c.startswith('PCA')] + ["Champion"]

# teams_df = teams_df.rename(columns={col: f"team_home_{col}" for col in cols_current})
# teams_df = teams_df.rename(columns={col: f"team_away_{col}" for col in cols_current})

for game_id, game_df in teams_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for location, team_df in game_df.groupby('Location'):
        # Team home
        if location == 'Home':
            prefix = "team_home"
            if team_df.iloc[0]["Win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "team_away"
        
        for col in pca_cols_current:
            if col == 'game_id' or col == 'Location' or col == 'Win' or col == "Opponent" or col == "team":
                continue
            row[f"{prefix}_{col}"] = team_df[col].values[0]
        
        team_historical_df = teams_historical_df[teams_historical_df['Team'] == team_df.iloc[0]['team']]
        for team_year in team_historical_df['Year'].unique():
            for col in pca_cols_historic:
                row[f"{prefix}_{str(team_year)}_historic_{col}"] = team_historical_df[col].values[0]
        
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)

if not os.path.exists(f"final_data/season_teams_dataset_{year}_reduced"):
    os.makedirs(f"final_data/season_teams_dataset_{year}_reduced")
# Guardar resultado
final_dataset.to_csv(f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Player DF reduced with context games

In [24]:
from collections import deque
teams = ["ATL", "BOS", "BRK", "CHI", "CHO", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"]
game_id_order = {}
for team in teams:
    games = {}
    context_games = deque(maxlen=4)
    for i in range(4):
        context_games.append(-1)
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    for game_id, game_location in zip(team_games_df['game_id'], team_games_df['Location']):
        games[game_id] = [game_location, context_games.copy()]
        context_games.append(game_id)
    game_id_order[team] = games


In [25]:
cola = deque(maxlen=4)
cola

deque([], maxlen=4)

In [26]:
"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)
context1_th_player1_component1_1, ..., context4_th_player7_componentM_Z, context1_ta_player1_component1_1, ..., context4_ta_player7_componentM_Z,

contextN = partidos anteriores de la misma temporada
"""
players_reduced_df = pd.read_csv(f"procesed_data/season_player_stats_{year}_reduced/all_players.csv", encoding='utf-8')
players_without_context_df = pd.read_csv(f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_{year}.csv", encoding='utf-8')
pca_cols_current = [c for c in players_reduced_df.columns if c.startswith('PCA')]

# Establecer el contexto de los partidos
prefixes = ['th', 'ta']
new_columns = {}

for prefix in prefixes:
    for i in range(1, 5):
        for player_idx in range(1, 8):
            for col in pca_cols_current:
                col_name = f"{prefix}_context{i}_player{player_idx}_{col}"
                new_columns[col_name] = np.zeros(len(players_without_context_df), dtype=float)

# Crear un DataFrame con las nuevas columnas
new_cols_df = pd.DataFrame(new_columns)

# Concatenar con el original
players_without_context_df = pd.concat([players_without_context_df, new_cols_df], axis=1)


for idx in range(len(players_without_context_df)):
    row = players_without_context_df.iloc[idx]
    game_id = row['game_id']

    for team, games_dict in game_id_order.items():
        if game_id in games_dict:
            game_location, context_games_original = games_dict[game_id]
            context_games = context_games_original.copy() 

            prefix = 'th' if game_location == 'Home' else 'ta'

            
            for i in range(1, 5):
                if not context_games:
                    continue
                context_game_id = context_games.pop()
                if context_game_id == -1:
                    continue
                for player_idx in range(1, 8):
                    # Extraer jugadores del partido de contexto
                    players_context = players_reduced_df[
                        (players_reduced_df['game_id'] == context_game_id) &
                        (players_reduced_df['location'] == game_location)
                    ].sort_values(by='mp', ascending=False)
                    # print(players_context)
                    player_row = players_context.iloc[player_idx - 1]

                    for col in pca_cols_current:
                        col_name = f"{prefix}_context{i}_player{player_idx}_{col}"
                        # print(col_name)
                        players_without_context_df.at[idx, col_name] = player_row[col]

players_without_context_df.to_csv(f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_with_context_{year}.csv", index=False, encoding='utf-8-sig')

### Player DF with context games

In [27]:
from collections import deque
teams = ["ATL", "BOS", "BRK", "CHI", "CHO", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"]
game_id_order = {}
for team in teams:
    games = {}
    context_games = deque(maxlen=4)
    for i in range(4):
        context_games.append(-1)
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    for game_id, game_location in zip(team_games_df['game_id'], team_games_df['Location']):
        games[game_id] = [game_location, context_games.copy()]
        context_games.append(game_id)
    game_id_order[team] = games


In [28]:
"""
th_player1_stat1, ..., th_player7_statZ, th_bench_usg, ..., ta_player1_stat1, ..., ta_player7_statZ, ta_bench_usg, ..., th_player1_historic_stat1, ..., 
th_player7_historic_statZ, ..., ta_player1_historic_stat1, ..., ta_player7_historic_statZ, team_home_wins (1 o 0)
context1_th_player1_stat1, ..., context4_th_player7_statZ, context1_ta_player1_stat1, ..., context4_ta_player7_statZ,

contextN = partidos anteriores de la misma temporada
"""
players_df = pd.read_csv(f"raw_data/season_player_stats_{year}/all_players.csv", encoding='utf-8')
players_without_context_df = pd.read_csv(f"final_data/season_players_dataset_{year}/season_players_dataset_{year}.csv", encoding='utf-8')
cols_current = [c for c in players_df.columns if not c in ["player","team","game_id","location","opponent","team_score","opponent_score","win","played","mp"]]

# Establecer el contexto de los partidos
prefixes = ['th', 'ta']
new_columns = {}

for prefix in prefixes:
    for i in range(1, 5):
        for player_idx in range(1, 8):
            for col in cols_current:
                col_name = f"{prefix}_context{i}_player{player_idx}_{col}"
                new_columns[col_name] = np.zeros(len(players_without_context_df), dtype=float)

# Crear un DataFrame con las nuevas columnas
new_cols_df = pd.DataFrame(new_columns)

# Concatenar con el original
players_without_context_df = pd.concat([players_without_context_df, new_cols_df], axis=1)


for idx in range(len(players_without_context_df)):
    row = players_without_context_df.iloc[idx]
    game_id = row['game_id']

    for team, games_dict in game_id_order.items():
        if game_id in games_dict:
            game_location, context_games_original = games_dict[game_id]
            context_games = context_games_original.copy() 

            prefix = 'th' if game_location == 'Home' else 'ta'

            
            for i in range(1, 5):
                if not context_games:
                    continue
                context_game_id = context_games.pop()
                if context_game_id == -1:
                    continue
                for player_idx in range(1, 8):
                    # Extraer jugadores del partido de contexto
                    players_context = players_df[
                        (players_df['game_id'] == context_game_id) &
                        (players_df['location'] == game_location)
                    ].sort_values(by='mp', ascending=False)
                    # print(players_context)
                    player_row = players_context.iloc[player_idx - 1]

                    for col in cols_current:
                        col_name = f"{prefix}_context{i}_player{player_idx}_{col}"
                        # print(col_name)
                        players_without_context_df.at[idx, col_name] = player_row[col]

players_without_context_df.to_csv(f"final_data/season_players_dataset_{year}/season_players_dataset_with_context_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF with context games

In [73]:
game_id_order = {}
for team in teams:
    games = {}
    context_games = deque(maxlen=4)
    for i in range(4):
        context_games.append(-1)
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    for game_id, game_location in zip(team_games_df['game_id'], team_games_df['Location']):
        games[game_id] = [game_location, context_games.copy()]
        context_games.append(game_id)
    game_id_order[team] = games

In [74]:
"""
team_home_stat1, ..., team_home_statN, team_home_historic_stat1, ..., team_home_historic_statZ, 
team_away_stat1, ..., team_away_statN, team_away_historic_stat1, ..., team_away_historic_statZ, team_home_wins (1 o 0)
context1_team_home_stat1, ..., context4_team_home_statN, context1_team_away_stat1, ..., context4_team_away_statN
"""

teams_df = pd.read_csv(f"raw_data/season_team_stats_{year}/all_teams.csv", encoding='utf-8')
teams_without_context_df = pd.read_csv(f"final_data/season_teams_dataset_{year}/season_teams_dataset_{year}.csv", encoding='utf-8')

cols_current = [c for c in teams_df.columns if not c.startswith('opp')]

# Establecer el contexto de los partidos
prefixes = ['team_home', 'team_away']
new_columns = {}

for prefix in prefixes:
    for i in range(1, 5):
        for col in cols_current:
            if col == 'game_id' or col == 'Location' or col == 'Win' or col == 'team' or col == "Opponent":
                continue
            col_name = f"{prefix}_context{i}_{col}"
            new_columns[col_name] = np.zeros(len(teams_without_context_df), dtype=float)

# Crear un DataFrame con las nuevas columnas
new_cols_df = pd.DataFrame(new_columns)

# Concatenar con el original
teams_without_context_df = pd.concat([teams_without_context_df, new_cols_df], axis=1)


for idx in range(len(teams_without_context_df)):
    row = teams_without_context_df.iloc[idx]
    game_id = row['game_id']

    for team, games_dict in game_id_order.items():
        if game_id in games_dict:
            game_location, context_games_original = games_dict[game_id]
            context_games = context_games_original.copy() 

            prefix = 'team_home' if game_location == 'Home' else 'team_away'

            
            for i in range(1, 5):
                if not context_games:
                    continue
                context_game_id = context_games.pop()
                if context_game_id == -1:
                    continue

                # Extraer jugadores del partido de contexto
                team_context = teams_df[(teams_df['game_id'] == context_game_id) & (teams_df['Location'] == game_location)]
                # print(players_context)
                team_row = team_context.iloc[0]

                for col in cols_current:
                    if col == 'game_id' or col == 'Location' or col == 'Win' or col == 'team' or col == "Opponent":
                        continue
                    col_name = f"{prefix}_context{i}_{col}"
                    # print(col_name)
                    teams_without_context_df.at[idx, col_name] = team_row[col]

teams_without_context_df.to_csv(f"final_data/season_teams_dataset_{year}/season_teams_dataset_with_context_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF reduced with context games

In [75]:
game_id_order = {}
for team in teams:
    games = {}
    context_games = deque(maxlen=4)
    for i in range(4):
        context_games.append(-1)
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    for game_id, game_location in zip(team_games_df['game_id'], team_games_df['Location']):
        games[game_id] = [game_location, context_games.copy()]
        context_games.append(game_id)
    game_id_order[team] = games

In [76]:
"""
team_home_component1_1, ..., team_home_componentM_Z, team_home_historic_component1_1, ..., team_home_historic_componentM_Z, 
team_away_component1_1, ..., team_away_componentM_Z, team_away_historic_component1_1, ..., team_away_historic_componentM_Z, team_home_wins (1 o 0)
context1_team_home_component1_1, ..., context4_team_home_componentM_Z, context1_team_away_component1_1, ..., context4_team_away_componentM_Z
"""

teams_df = pd.read_csv(f"procesed_data/season_team_stats_{year}_reduced/all_teams.csv", encoding='utf-8')
teams_without_context_df = pd.read_csv(f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_{year}.csv", encoding='utf-8')

pca_cols_current = [c for c in teams_df.columns if not (c.startswith('PCA6') or c.startswith('PCA7') or c.startswith('PCA8'))]

# Establecer el contexto de los partidos
prefixes = ['team_home', 'team_away']
new_columns = {}

for prefix in prefixes:
    for i in range(1, 5):
        for col in pca_cols_current:
            if col == 'game_id' or col == 'Location' or col == 'Win' or col == 'team' or col == "Opponent":
                continue
            col_name = f"{prefix}_context{i}_{col}"
            new_columns[col_name] = np.zeros(len(teams_without_context_df), dtype=float)

# Crear un DataFrame con las nuevas columnas
new_cols_df = pd.DataFrame(new_columns)

# Concatenar con el original
teams_without_context_df = pd.concat([teams_without_context_df, new_cols_df], axis=1)


for idx in range(len(teams_without_context_df)):
    row = teams_without_context_df.iloc[idx]
    game_id = row['game_id']

    for team, games_dict in game_id_order.items():
        if game_id in games_dict:
            game_location, context_games_original = games_dict[game_id]
            context_games = context_games_original.copy() 

            prefix = 'team_home' if game_location == 'Home' else 'team_away'

            
            for i in range(1, 5):
                if not context_games:
                    continue
                context_game_id = context_games.pop()
                if context_game_id == -1:
                    continue

                # Extraer jugadores del partido de contexto
                team_context = teams_df[(teams_df['game_id'] == context_game_id) & (teams_df['Location'] == game_location)]
                # print(players_context)
                team_row = team_context.iloc[0]

                for col in pca_cols_current:
                    if col == 'game_id' or col == 'Location' or col == 'Win' or col == 'team' or col == "Opponent":
                        continue
                    col_name = f"{prefix}_context{i}_{col}"
                    # print(col_name)
                    teams_without_context_df.at[idx, col_name] = team_row[col]

teams_without_context_df.to_csv(f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_with_context_{year}.csv", index=False, encoding='utf-8-sig')

# Test datasets

In [77]:
teams = ["ATL", "BOS", "BRK", "CHI", "CHO", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"]
last_game_by_team = {}
for team in teams:
    team_games_df = pd.read_csv(f"raw_data/season_team_stats_{year}/{team}.csv", encoding='utf-8')
    last_game = team_games_df.iloc[-1]
    last_game_by_team[team] = {
        "game_id": last_game["game_id"],
        "Location": last_game["Location"]
    }

last_game_by_team

{'ATL': {'game_id': '94b63567-8ff5-4ff5-b197-ad8984a36fce',
  'Location': 'Away'},
 'BOS': {'game_id': '596635cb-408d-4665-a13b-cb39895904ab',
  'Location': 'Home'},
 'BRK': {'game_id': '7841dde3-dcef-467f-a86e-9c741c923a69',
  'Location': 'Away'},
 'CHI': {'game_id': '8bb4f889-96eb-4ef1-9dc0-9e5f77a50314',
  'Location': 'Away'},
 'CHO': {'game_id': 'f7c804f1-8282-4dab-b9ac-9a566943dfba',
  'Location': 'Away'},
 'CLE': {'game_id': 'f7c804f1-8282-4dab-b9ac-9a566943dfba',
  'Location': 'Home'},
 'DAL': {'game_id': '8cdf11e8-a88f-4921-ab68-627bfd66aafb',
  'Location': 'Away'},
 'DEN': {'game_id': 'ca351999-a837-4454-a878-2e0e21b31987',
  'Location': 'Away'},
 'DET': {'game_id': '2f9c0f37-d2a3-4e10-9dbc-db4d798b5672',
  'Location': 'Away'},
 'GSW': {'game_id': 'eb2ec681-fb83-49cc-b06f-092369c120b5',
  'Location': 'Home'},
 'HOU': {'game_id': 'eca3db22-6f6d-4fbf-8ea4-7eccf7af6f17',
  'Location': 'Away'},
 'IND': {'game_id': '94b63567-8ff5-4ff5-b197-ad8984a36fce',
  'Location': 'Home'},
 'LA

### Player DF reduced with context games

In [ ]:
"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)
context1_th_player1_component1_1, ..., context4_th_player7_componentM_Z, context1_ta_player1_component1_1, ..., context4_ta_player7_componentM_Z,

contextN = partidos anteriores de la misma temporada
"""

players_reduced_training_df = pd.read_csv(f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_with_context_{year}.csv", encoding='utf-8-sig')
cols = [c for c in players_reduced_training_df.columns if c.startswith('th')]
cols = [c.replace("th_", "") for c in cols]
test_dataset = pd.DataFrame(columns = cols + ["team"])

for team, last_game in last_game_by_team.items():
	game = players_reduced_training_df[players_reduced_training_df["game_id"] == last_game["game_id"]]
	game = game.reset_index(drop=True)
	if last_game["Location"] == "Home":
		prefix = "th_"
	else:
		prefix = "ta_"
	final_game_stats = {}
	final_game_stats["team"] = team
	for col in cols:
		final_game_stats[col] = game.loc[0, (prefix + col)]
	test_dataset.loc[len(test_dataset)] = final_game_stats

if not os.path.exists(f"final_data/season_players_dataset_{year}_reduced/test_dataset"):
    os.makedirs(f"final_data/season_players_dataset_{year}_reduced/test_dataset")
test_dataset.to_csv(f"final_data/season_players_dataset_{year}_reduced/test_dataset/test_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Player DF with context games

In [ ]:
players_training_df = pd.read_csv(f"final_data/season_players_dataset_{year}/season_players_dataset_with_context_{year}.csv", encoding='utf-8-sig')
cols = [c for c in players_training_df.columns if c.startswith('th')]
cols = [c.replace("th_", "") for c in cols]
test_dataset = pd.DataFrame(columns = cols + ["team"])

for team, last_game in last_game_by_team.items():
	game = players_training_df[players_training_df["game_id"] == last_game["game_id"]]
	game = game.reset_index(drop=True)
	if last_game["Location"] == "Home":
		prefix = "th_"
	else:
		prefix = "ta_"
	final_game_stats = {}
	final_game_stats["team"] = team
	for col in cols:
		final_game_stats[col] = game.loc[0, (prefix + col)]
	test_dataset.loc[len(test_dataset)] = final_game_stats

if not os.path.exists(f"final_data/season_players_dataset_{year}/test_dataset"):
    os.makedirs(f"final_data/season_players_dataset_{year}/test_dataset")
test_dataset.to_csv(f"final_data/season_players_dataset_{year}/test_dataset/test_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF with context games

In [ ]:
teams_training_df = pd.read_csv(f"final_data/season_teams_dataset_{year}/season_teams_dataset_with_context_{year}.csv", encoding='utf-8-sig')
cols = [c for c in teams_training_df.columns if c.startswith('team_home') and not c == "team_home_wins"]
cols = [c.replace("team_home_", "") for c in cols]
test_dataset = pd.DataFrame(columns = cols + ["team"])

for team, last_game in last_game_by_team.items():
	game = teams_training_df[teams_training_df["game_id"] == last_game["game_id"]]
	game = game.reset_index(drop=True)
	if last_game["Location"] == "Home":
		prefix = "team_home_"
	else:
		prefix = "team_away_"
	final_game_stats = {}
	final_game_stats["team"] = team
	for col in cols:
		final_game_stats[col] = game.loc[0, (prefix + col)]
	test_dataset.loc[len(test_dataset)] = final_game_stats

if not os.path.exists(f"final_data/season_teams_dataset_{year}/test_dataset"):
    os.makedirs(f"final_data/season_teams_dataset_{year}/test_dataset")
test_dataset.to_csv(f"final_data/season_teams_dataset_{year}/test_dataset/test_dataset_{year}.csv", index=False, encoding='utf-8-sig')

### Teams DF reduced with context games

In [ ]:
teams_reduced_training_df = pd.read_csv(f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_with_context_{year}.csv", encoding='utf-8-sig')
cols = [c for c in teams_reduced_training_df.columns if c.startswith('team_home') and not c == "team_home_wins"]
cols = [c.replace("team_home_", "") for c in cols]
test_dataset = pd.DataFrame(columns = cols + ["team"])

for team, last_game in last_game_by_team.items():
	game = teams_reduced_training_df[teams_reduced_training_df["game_id"] == last_game["game_id"]]
	game = game.reset_index(drop=True)
	if last_game["Location"] == "Home":
		prefix = "team_home_"
	else:
		prefix = "team_away_"
	final_game_stats = {}
	final_game_stats["team"] = team
	for col in cols:
		final_game_stats[col] = game.loc[0, (prefix + col)]
	test_dataset.loc[len(test_dataset)] = final_game_stats

if not os.path.exists(f"final_data/season_teams_dataset_{year}_reduced/test_dataset"):
    os.makedirs(f"final_data/season_teams_dataset_{year}_reduced/test_dataset")
test_dataset.to_csv(f"final_data/season_teams_dataset_{year}_reduced/test_dataset/test_dataset_{year}.csv", index=False, encoding='utf-8-sig')